In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import col, to_date

data = [
    ("Alice",   "IN",  "2024-01-01", 5000),
    ("Bob",     "IN",  "2024-01-02", 3000),
    ("Charlie", "IN",  "2024-01-03", 7000),
    ("Diana",   "IN",  "2024-01-04", 4000),
    ("Eve",     "US",  "2024-01-01", 8000),
    ("Frank",   "US",  "2024-01-02", 6000),
    ("Grace",   "US",  "2024-01-03", 9000),
    ("Hank",    "US",  "2024-01-04", 2000),
    ("Ivy",     "EU",  "2024-01-01", 4500),
    ("Jack",    "EU",  "2024-01-02", 7500),
    ("Karen",   "EU",  "2024-01-03", 3500),
    ("Leo",     "EU",  "2024-01-04", 6500),
]

schema = StructType([
    StructField("name",    StringType(),  True),
    StructField("region",  StringType(),  True),
    StructField("date",    StringType(),  True),
    StructField("sales",   IntegerType(), True),
])

df = spark.createDataFrame(data, schema) \
          .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number, rank, dense_rank,
    lag, lead, sum, avg, min, max
)

# Window — partition by region, order by sales descending
w_rank  = Window.partitionBy("region").orderBy(col("sales").desc())

# Window — partition by region, order by date (for running totals/lag/lead)
w_date  = Window.partitionBy("region").orderBy("date")

# Window — running total (from first row to current row)
w_running = Window.partitionBy("region").orderBy("date") \
                  .rowsBetween(Window.unboundedPreceding, 0)

In [0]:
df.withColumn("row_number",  row_number().over(w_rank)) \
  .withColumn("rank",        rank().over(w_rank)) \
  .withColumn("dense_rank",  dense_rank().over(w_rank)) \
  .select("name", "region", "sales", "row_number", "rank", "dense_rank") \
  .show()

In [0]:
df.withColumn("prev_sales", lag("sales",  1).over(w_date)) \
  .withColumn("next_sales", lead("sales", 1).over(w_date)) \
  .select("name", "region", "date", "sales", "prev_sales", "next_sales") \
  .show()

In [0]:
df.withColumn("running_total", sum("sales").over(w_running)) \
  .withColumn("region_avg",    avg("sales").over(Window.partitionBy("region"))) \
  .withColumn("region_max",    max("sales").over(Window.partitionBy("region"))) \
  .select("name", "region", "date", "sales", "running_total", "region_avg", "region_max") \
  .show()

![image_1775202379706.png](./image_1775202379706.png "image_1775202379706.png")

In [0]:
data = [
    ("Alice",   "IN", 7000),
    ("Bob",     "IN", 7000),    # tie with Alice
    ("Karen",   "IN", 7000),    # tie with Alice & Bob   
    ("Charlie", "IN", 5000),
    ("Diana",   "IN", 5000),   # tie with Charlie
    ("Eve",     "IN", 3000),
]
df1 = spark.createDataFrame(data, ["name", "region", "sales"])

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number, col

w = Window.partitionBy("region").orderBy(col("sales").desc())

df1.withColumn("rank",       rank().over(w)) \
  .withColumn("dense_rank", dense_rank().over(w)) \
  .withColumn("row_number", row_number().over(w)) \
  .show()

![image_1775202176547.png](./image_1775202176547.png "image_1775202176547.png")


 Top 2 Sales Per Region

In [0]:
w = Window.partitionBy("region").orderBy(col("sales").desc())
# With row_number — always exactly 2 rows per region
df.withColumn("rn", row_number().over(w)) \
  .filter(col("rn") <= 2).show()    # guaranteed exactly 2 rows

# With rank — may return MORE than 2 rows if there's a tie at position 2
df.withColumn("rn", rank().over(w)) \
  .filter(col("rn") <= 2).show() # could return 3 rows if two people tie for rank 2